# Контроль заполнения обязательных данных по версиям моделей в эксплуатации

Ноутбук формирует отчёт `model_required_fields_gaps.xlsx`.

**Периметр:** актуальные версии моделей, для которых `model_ver_prom_expl_flag = true`.

**Срез:** текущее состояние данных.

**Результат:**
- лист `Детализация` — компактная таблица: модель, название, версия, дата ввода версии, блок/подразделение разработчика и владельца, степень значимости, затем флаги по каждому проверяемому реквизиту;
- лист `Статистика` — количество и доля версий с нарушениями по каждому реквизиту.

Флаги в Excel:
- `ОК`;
- `Нарушение`;
- `Не применимо`.

Для финансово-значимых реквизитов отдельного справочника нет, поэтому этот пункт не может быть проверен автоматически и в выгрузке помечается как `Не применимо`.


In [ ]:
import os
import sys

os.environ["SPARK_MAJOR_VERSION"] = "3.5.1"
os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3.5.1-client/"
os.environ["PYSPARK_DRIVER"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/")
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/lib/py4j-0.10.9.7-src.zip")

from pathlib import Path

import pandas as pd
from pyspark import SparkConf
from pyspark.sql import DataFrame, SparkSession, Window, functions as F

conf = (
    SparkConf()
    .setAppName("model_required_fields_gaps")
    .setMaster("yarn")
    .set("spark.executor.cores", "2")
    .set("spark.executor.memory", "6g")
    .set("spark.executor.memoryOverhead", "1g")
    .set("spark.driver.memory", "6g")
    .set("spark.driver.maxResultSize", "4g")
    .set("spark.shuffle.service.enabled", "true")
    .set("spark.dynamicAllocation.enabled", "true")
    .set("spark.dynamicAllocation.initialExecutors", "4")
    .set("spark.dynamicAllocation.maxExecutors", "12")
    .set("spark.dynamicAllocation.executorIdleTimeout", "120s")
    .set("spark.dynamicAllocation.cachedExecutorIdleTimeout", "600s")
    .set("spark.dynamicAllocation.shuffleTracking.enabled", "true")
    .set("spark.port.maxRetries", "150")
)

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)


In [ ]:
SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"

OUTPUT_DIR = Path.cwd().resolve()
OUTPUT_FILE = OUTPUT_DIR / "model_required_fields_gaps.xlsx"
MAX_EXCEL_ROWS = 1_048_575

TABLES = {
    "model": f"{SOURCE_DB}.t_model",
    "model_ver": f"{SOURCE_DB}.t_model_ver",
    "model_ver_anlt_dtl": f"{SOURCE_DB}.t_model_ver_anlt_dtl",
    "valid": f"{SOURCE_DB}.t_valid",
    "model_ver_x_montrg_manual": f"{SOURCE_DB}.t_model_ver_x_montrg_manual",
    "montrg_manual_rslt": f"{SOURCE_DB}.t_montrg_manual_rslt",
    "model_ver_x_montrg_auto": f"{SOURCE_DB}.t_model_ver_x_montrg_auto",
    "montrg_auto_rslt": f"{SOURCE_DB}.t_montrg_auto_rslt",
    "sample_data": f"{SOURCE_DB}.t_sample_data",
    "change_log": f"{SOURCE_DB}.t_ent_param_chg",
}

STATUS_OK = "OK"
STATUS_MISSING = "НЕТ ДАННЫХ"
STATUS_EXCEPTION = "НЕ ПРИМЕНИМО / ИСКЛЮЧЕНИЕ"
STATUS_NOT_CHECKED = "НЕ ПРОВЕРЕНО"


## Служебные функции

In [ ]:
def require_columns(df: DataFrame, table_name: str, columns) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise RuntimeError(f"{table_name}: отсутствуют поля: {', '.join(missing)}")


def without_deleted(df: DataFrame) -> DataFrame:
    if "ctl_action" not in df.columns:
        return df
    return df.filter(
        F.col("ctl_action").isNull()
        | (F.upper(F.trim(F.col("ctl_action").cast("string"))) != F.lit("D"))
    )


def take_latest(df: DataFrame, keys, order_column: str) -> DataFrame:
    window = Window.partitionBy(*keys).orderBy(F.col(order_column).desc_nulls_last())
    return (
        df.withColumn("__rn", F.row_number().over(window))
        .filter(F.col("__rn") == 1)
        .drop("__rn")
    )


def has_value(column_name: str):
    c = F.col(column_name)
    s = F.lower(F.trim(c.cast("string")))
    return (
        c.isNotNull()
        & (s != "")
        & (~s.isin("null", "none", "nan", "[]", "{}"))
    )


def is_not_applicable_marker(column_name: str):
    s = F.lower(F.trim(F.col(column_name).cast("string")))
    return s.isin(
        "$not_applicable$",
        "not applicable",
        "не применимо",
        "неприменимо",
        "n/a",
        "na",
    )


def truthy(column_name: str):
    s = F.lower(F.trim(F.col(column_name).cast("string")))
    return s.isin("true", "1", "yes", "y", "да")


def status_for_value(column_name: str, allow_not_applicable=True):
    if allow_not_applicable:
        return (
            F.when(is_not_applicable_marker(column_name), F.lit(STATUS_EXCEPTION))
            .when(has_value(column_name), F.lit(STATUS_OK))
            .otherwise(F.lit(STATUS_MISSING))
        )
    return F.when(has_value(column_name), F.lit(STATUS_OK)).otherwise(F.lit(STATUS_MISSING))


def int_flag(condition):
    return F.when(condition, F.lit(1)).otherwise(F.lit(0))


## Чтение источников и проверка схемы

In [ ]:
for table_name in TABLES.values():
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Не найдена таблица: {table_name}")

model_raw = spark.table(TABLES["model"])
model_ver_raw = spark.table(TABLES["model_ver"])
anlt_raw = spark.table(TABLES["model_ver_anlt_dtl"])
valid_raw = spark.table(TABLES["valid"])
manual_link_raw = spark.table(TABLES["model_ver_x_montrg_manual"])
manual_result_raw = spark.table(TABLES["montrg_manual_rslt"])
auto_link_raw = spark.table(TABLES["model_ver_x_montrg_auto"])
auto_result_raw = spark.table(TABLES["montrg_auto_rslt"])
sample_raw = spark.table(TABLES["sample_data"])
change_raw = spark.table(TABLES["change_log"])

required = {
    TABLES["model"]: [
        "model_sid", "model_name", "model_code", "model_other_docs_sid", "start_dt"
    ],
    TABLES["model_ver"]: [
        "model_sid", "model_ver_sid", "model_ver_num", "model_ver_name",
        "model_ver_crtn_dttm",
        "model_ver_signfcnt_ctgry_code", "model_ver_signfcnt_lvl_name",
        "model_ver_other_docs_sid", "model_ver_dev_report_sid",
        "model_ver_repstry_link_txt", "model_ver_implm_repstry_link_txt",
        "model_ver_repstry_non_link_reason_name",
        "model_ver_repstry_non_link_reason_cmnt_txt",
        "model_ver_montrg_auto_exclude_reason_array",
        "start_dt",
    ],
    TABLES["model_ver_anlt_dtl"]: [
        "model_ver_sid", "model_ver_stts_name", "model_stts_name",
        "model_ver_prom_expl_flag",
        "model_ver_prom_actual_implm_first_dttm",
        "model_ver_prom_implm_first_dttm",
        "model_ver_prom_actual_expl_output_dttm",
        "model_ver_prom_expl_output_dttm",
        "model_ver_owner_block_name", "model_ver_owner_dprtmt_name",
        "model_ver_owner_cont_array",
        "model_ver_dev_block_name", "model_ver_dev_dprtmt_name",
        "model_ver_dev_dprtmt_cont_array",
        "model_ver_employer_block_name", "model_ver_employer_dprtmt_name",
        "model_ver_risk_sgmnt_name", "model_ver_busn_proc_code",
        "model_ver_signfcnt_ctgry_code", "model_ver_signfcnt_lvl_name",
        "model_ver_not_valid_flag",
        "start_dt",
    ],
    TABLES["valid"]: [
        "model_ver_sid", "valid_sid", "valid_rslt_name", "valid_report_sid",
        "valid_not_sample_data_flag",
    ],
    TABLES["model_ver_x_montrg_manual"]: [
        "model_ver_sid", "montrg_manual_sid"
    ],
    TABLES["montrg_manual_rslt"]: [
        "montrg_manual_sid", "montrg_manual_rslt_sid",
        "montrg_manual_rslt_name", "montrg_manual_rslt_report_sid",
        "montrg_manual_rslt_valid_not_sample_data_flag",
    ],
    TABLES["model_ver_x_montrg_auto"]: [
        "model_ver_sid", "montrg_auto_sid"
    ],
    TABLES["montrg_auto_rslt"]: [
        "montrg_auto_sid", "montrg_auto_rslt_sid",
        "montrg_auto_rslt_name", "montrg_auto_rslt_report_sid",
        "montrg_auto_rslt_valid_not_sample_data_flag",
    ],
    TABLES["sample_data"]: [
        "model_ver_sid", "valid_sid",
        "montrg_manual_rslt_sid", "montrg_auto_rslt_sid",
        "sample_data_addtnl_info_txt",
    ],
    TABLES["change_log"]: [
        "ent_sid", "ent_type_name", "ent_param_chg_sid",
        "ent_param_chg_name", "ctl_action"
    ],
}

for table_name, columns in required.items():
    require_columns(spark.table(table_name), table_name, columns)

print("Все таблицы и обязательные поля найдены")


## Актуальный периметр версий в эксплуатации

В отличие от старой логики, актуальная запись `T_MODEL_VER` выбирается **по `model_ver_sid`**, а не по `model_sid`. Это сохраняет все самостоятельные версии одной модели.


In [ ]:
model_current = (
    take_latest(without_deleted(model_raw), ["model_sid"], "start_dt")
    .select(
        F.col("model_sid").cast("string").alias("model_sid"),
        "model_name",
        "model_code",
        "model_other_docs_sid",
    )
)

model_ver_current = (
    take_latest(without_deleted(model_ver_raw), ["model_ver_sid"], "start_dt")
    .select(
        F.col("model_sid").cast("string").alias("model_sid"),
        F.col("model_ver_sid").cast("string").alias("model_ver_sid"),
        "model_ver_num",
        "model_ver_name",
        "model_ver_crtn_dttm",
        "model_ver_signfcnt_ctgry_code",
        "model_ver_signfcnt_lvl_name",
        "model_ver_other_docs_sid",
        "model_ver_dev_report_sid",
        "model_ver_repstry_link_txt",
        "model_ver_implm_repstry_link_txt",
        "model_ver_repstry_non_link_reason_name",
        "model_ver_repstry_non_link_reason_cmnt_txt",
        "model_ver_montrg_auto_exclude_reason_array",
    )
)

anlt_current = (
    take_latest(without_deleted(anlt_raw), ["model_ver_sid"], "start_dt")
    .select(
        F.col("model_ver_sid").cast("string").alias("model_ver_sid"),
        "model_ver_stts_name",
        "model_stts_name",
        "model_ver_prom_expl_flag",
        "model_ver_prom_actual_implm_first_dttm",
        "model_ver_prom_implm_first_dttm",
        "model_ver_prom_actual_expl_output_dttm",
        "model_ver_prom_expl_output_dttm",
        "model_ver_owner_block_name",
        "model_ver_owner_dprtmt_name",
        "model_ver_owner_cont_array",
        "model_ver_dev_block_name",
        "model_ver_dev_dprtmt_name",
        "model_ver_dev_dprtmt_cont_array",
        "model_ver_employer_block_name",
        "model_ver_employer_dprtmt_name",
        "model_ver_risk_sgmnt_name",
        "model_ver_busn_proc_code",
        F.col("model_ver_signfcnt_ctgry_code").alias("anlt_signfcnt_ctgry_code"),
        F.col("model_ver_signfcnt_lvl_name").alias("anlt_signfcnt_lvl_name"),
        "model_ver_not_valid_flag",
    )
)

version_context = (
    model_ver_current.alias("v")
    .join(model_current.alias("m"), "model_sid", "inner")
    .join(anlt_current.alias("a"), "model_ver_sid", "left")
    .filter(F.col("model_ver_prom_expl_flag").cast("boolean") == F.lit(True))
    .select(
        "model_sid",
        "model_ver_sid",
        "model_code",
        "model_name",
        "model_ver_num",
        "model_ver_name",
        "model_ver_crtn_dttm",
        F.col("model_ver_stts_name").alias("current_model_ver_status"),
        F.col("model_stts_name").alias("current_model_status"),
        F.coalesce(
            F.col("model_ver_prom_actual_implm_first_dttm"),
            F.col("model_ver_prom_implm_first_dttm"),
        ).alias("operation_start_dttm"),
        F.coalesce(
            F.col("model_ver_prom_actual_expl_output_dttm"),
            F.col("model_ver_prom_expl_output_dttm"),
        ).alias("operation_end_dttm"),
        "model_ver_prom_expl_flag",
        "model_ver_owner_block_name",
        "model_ver_owner_dprtmt_name",
        F.col("model_ver_owner_cont_array").cast("string").alias("model_ver_owner_cont_array"),
        "model_ver_dev_block_name",
        "model_ver_dev_dprtmt_name",
        F.col("model_ver_dev_dprtmt_cont_array").cast("string").alias("model_ver_dev_dprtmt_cont_array"),
        "model_ver_employer_block_name",
        "model_ver_employer_dprtmt_name",
        "model_ver_risk_sgmnt_name",
        "model_ver_busn_proc_code",
        F.coalesce(
            F.col("anlt_signfcnt_ctgry_code"),
            F.col("model_ver_signfcnt_ctgry_code"),
        ).alias("model_ver_signfcnt_ctgry_code"),
        F.coalesce(
            F.col("anlt_signfcnt_lvl_name"),
            F.col("model_ver_signfcnt_lvl_name"),
        ).alias("model_ver_signfcnt_lvl_name"),
        "model_ver_dev_report_sid",
        "model_ver_other_docs_sid",
        "model_other_docs_sid",
        "model_ver_repstry_link_txt",
        "model_ver_implm_repstry_link_txt",
        "model_ver_repstry_non_link_reason_name",
        "model_ver_repstry_non_link_reason_cmnt_txt",
        F.col("model_ver_montrg_auto_exclude_reason_array").cast("string").alias(
            "model_ver_montrg_auto_exclude_reason_array"
        ),
        "model_ver_not_valid_flag",
    )
    .cache()
)

operational_count = version_context.count()
if operational_count == 0:
    raise RuntimeError("Не найдено версий моделей с model_ver_prom_expl_flag = true")

print("Версий моделей в эксплуатации:", operational_count)
print("Моделей в эксплуатации:", version_context.select("model_sid").distinct().count())


## Валидация, ручной и автоматический мониторинг

In [ ]:
valid_effective = without_deleted(valid_raw)

validation_agg = (
    valid_effective
    .withColumn("model_ver_sid", F.col("model_ver_sid").cast("string"))
    .groupBy("model_ver_sid")
    .agg(
        F.countDistinct("valid_sid").alias("validation_count"),
        F.max(
            int_flag(has_value("valid_rslt_name") | has_value("valid_report_sid"))
        ).alias("validation_result_present"),
        F.max(
            int_flag(truthy("valid_not_sample_data_flag"))
        ).alias("validation_no_sample_exception"),
        F.concat_ws(
            " | ",
            F.collect_set(F.col("valid_rslt_name").cast("string")),
        ).alias("validation_results"),
        F.concat_ws(
            " | ",
            F.collect_set(F.col("valid_report_sid").cast("string")),
        ).alias("validation_reports"),
    )
)

manual_links = (
    without_deleted(manual_link_raw)
    .select(
        F.col("model_ver_sid").cast("string").alias("model_ver_sid"),
        F.col("montrg_manual_sid").cast("string").alias("montrg_manual_sid"),
    )
)

manual_results = (
    without_deleted(manual_result_raw)
    .select(
        F.col("montrg_manual_sid").cast("string").alias("montrg_manual_sid"),
        F.col("montrg_manual_rslt_sid").cast("string").alias("montrg_manual_rslt_sid"),
        "montrg_manual_rslt_name",
        "montrg_manual_rslt_report_sid",
        "montrg_manual_rslt_valid_not_sample_data_flag",
    )
)

manual_agg = (
    manual_links.alias("l")
    .join(manual_results.alias("r"), "montrg_manual_sid", "left")
    .groupBy("model_ver_sid")
    .agg(
        F.countDistinct("montrg_manual_rslt_sid").alias("manual_monitoring_result_count"),
        F.max(
            int_flag(
                has_value("montrg_manual_rslt_name")
                | has_value("montrg_manual_rslt_report_sid")
            )
        ).alias("manual_monitoring_result_present"),
        F.max(
            int_flag(truthy("montrg_manual_rslt_valid_not_sample_data_flag"))
        ).alias("manual_monitoring_no_sample_exception"),
        F.concat_ws(
            " | ",
            F.collect_set(F.col("montrg_manual_rslt_name").cast("string")),
        ).alias("manual_monitoring_results"),
        F.concat_ws(
            " | ",
            F.collect_set(F.col("montrg_manual_rslt_report_sid").cast("string")),
        ).alias("manual_monitoring_reports"),
    )
)

auto_links = (
    without_deleted(auto_link_raw)
    .select(
        F.col("model_ver_sid").cast("string").alias("model_ver_sid"),
        F.col("montrg_auto_sid").cast("string").alias("montrg_auto_sid"),
    )
)

auto_results = (
    without_deleted(auto_result_raw)
    .select(
        F.col("montrg_auto_sid").cast("string").alias("montrg_auto_sid"),
        F.col("montrg_auto_rslt_sid").cast("string").alias("montrg_auto_rslt_sid"),
        "montrg_auto_rslt_name",
        "montrg_auto_rslt_report_sid",
        "montrg_auto_rslt_valid_not_sample_data_flag",
    )
)

auto_agg = (
    auto_links.alias("l")
    .join(auto_results.alias("r"), "montrg_auto_sid", "left")
    .groupBy("model_ver_sid")
    .agg(
        F.countDistinct("montrg_auto_rslt_sid").alias("auto_monitoring_result_count"),
        F.max(
            int_flag(
                has_value("montrg_auto_rslt_name")
                | has_value("montrg_auto_rslt_report_sid")
            )
        ).alias("auto_monitoring_result_present"),
        F.max(
            int_flag(truthy("montrg_auto_rslt_valid_not_sample_data_flag"))
        ).alias("auto_monitoring_no_sample_exception"),
        F.concat_ws(
            " | ",
            F.collect_set(F.col("montrg_auto_rslt_name").cast("string")),
        ).alias("auto_monitoring_results"),
        F.concat_ws(
            " | ",
            F.collect_set(F.col("montrg_auto_rslt_report_sid").cast("string")),
        ).alias("auto_monitoring_reports"),
    )
)


## Ссылки/пути к данным валидации и мониторинга

In [ ]:
sample_effective = without_deleted(sample_raw)

sample_agg = (
    sample_effective
    .withColumn("model_ver_sid", F.col("model_ver_sid").cast("string"))
    .groupBy("model_ver_sid")
    .agg(
        F.max(
            int_flag(
                has_value("valid_sid")
                & has_value("sample_data_addtnl_info_txt")
            )
        ).alias("validation_data_present"),
        F.max(
            int_flag(
                has_value("montrg_manual_rslt_sid")
                & has_value("sample_data_addtnl_info_txt")
            )
        ).alias("manual_monitoring_data_present"),
        F.max(
            int_flag(
                has_value("montrg_auto_rslt_sid")
                & has_value("sample_data_addtnl_info_txt")
            )
        ).alias("auto_monitoring_data_present"),
        F.concat_ws(
            " | ",
            F.collect_set(
                F.when(
                    has_value("valid_sid"),
                    F.col("sample_data_addtnl_info_txt").cast("string"),
                )
            ),
        ).alias("validation_data_links"),
        F.concat_ws(
            " | ",
            F.collect_set(
                F.when(
                    has_value("montrg_manual_rslt_sid"),
                    F.col("sample_data_addtnl_info_txt").cast("string"),
                )
            ),
        ).alias("manual_monitoring_data_links"),
        F.concat_ws(
            " | ",
            F.collect_set(
                F.when(
                    has_value("montrg_auto_rslt_sid"),
                    F.col("sample_data_addtnl_info_txt").cast("string"),
                )
            ),
        ).alias("auto_monitoring_data_links"),
    )
)


## История изменений

Проверяются изменения как самой модели (`MODEL`), так и версии модели (`MODEL_VERSION` / `MODEL_VER`).

Финансово-значимые реквизиты отдельно не выделяются: в доступной схеме отсутствует справочник, связывающий `ENT_PARAM_CHG_SID` с признаком финансовой значимости.


In [ ]:
change_effective = without_deleted(change_raw)

version_change_agg = (
    change_effective
    .filter(
        F.upper(F.col("ent_param_chg_sid").cast("string")).startswith("MODEL_VERSION_")
        | F.upper(F.col("ent_type_name").cast("string")).isin("MODEL_VERSION", "MODEL_VER")
    )
    .withColumn("model_ver_sid", F.col("ent_sid").cast("string"))
    .groupBy("model_ver_sid")
    .agg(F.count(F.lit(1)).alias("version_change_count"))
)

model_change_agg = (
    change_effective
    .filter(
        F.upper(F.col("ent_type_name").cast("string")) == F.lit("MODEL")
    )
    .withColumn("model_sid", F.col("ent_sid").cast("string"))
    .groupBy("model_sid")
    .agg(F.count(F.lit(1)).alias("model_change_count"))
)


## Проверка обязательных полей

In [ ]:
detail = (
    version_context
    .join(validation_agg, "model_ver_sid", "left")
    .join(manual_agg, "model_ver_sid", "left")
    .join(auto_agg, "model_ver_sid", "left")
    .join(sample_agg, "model_ver_sid", "left")
    .join(version_change_agg, "model_ver_sid", "left")
    .join(model_change_agg, "model_sid", "left")
    .fillna(
        {
            "validation_count": 0,
            "validation_result_present": 0,
            "validation_no_sample_exception": 0,
            "manual_monitoring_result_count": 0,
            "manual_monitoring_result_present": 0,
            "manual_monitoring_no_sample_exception": 0,
            "auto_monitoring_result_count": 0,
            "auto_monitoring_result_present": 0,
            "auto_monitoring_no_sample_exception": 0,
            "validation_data_present": 0,
            "manual_monitoring_data_present": 0,
            "auto_monitoring_data_present": 0,
            "version_change_count": 0,
            "model_change_count": 0,
        }
    )
)

detail = (
    detail
    # Код + название модели.
    .withColumn(
        "status_model_identification",
        F.when(
            has_value("model_code") & has_value("model_name"),
            F.lit(STATUS_OK),
        ).otherwise(F.lit(STATUS_MISSING)),
    )
    # Дата разработки/корректировки: по согласованной логике используется дата версии.
    .withColumn(
        "status_model_version_date",
        status_for_value("model_ver_crtn_dttm", allow_not_applicable=False),
    )
    # Начало эксплуатации обязательно.
    .withColumn(
        "status_operation_start",
        status_for_value("operation_start_dttm", allow_not_applicable=False),
    )
    # Для версии, которая сейчас в эксплуатации, отсутствие даты окончания допустимо.
    .withColumn(
        "status_operation_end",
        F.when(has_value("operation_end_dttm"), F.lit(STATUS_OK))
        .when(
            F.col("model_ver_prom_expl_flag").cast("boolean") == F.lit(True),
            F.lit(STATUS_EXCEPTION),
        )
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # Владелец: контакт + подразделение + блок.
    .withColumn(
        "status_owner",
        F.when(
            has_value("model_ver_owner_cont_array")
            & has_value("model_ver_owner_dprtmt_name")
            & has_value("model_ver_owner_block_name"),
            F.lit(STATUS_OK),
        ).otherwise(F.lit(STATUS_MISSING)),
    )
    # Разработчик: команда + подразделение + блок.
    .withColumn(
        "status_developer",
        F.when(
            has_value("model_ver_dev_dprtmt_cont_array")
            & has_value("model_ver_dev_dprtmt_name")
            & has_value("model_ver_dev_block_name"),
            F.lit(STATUS_OK),
        ).otherwise(F.lit(STATUS_MISSING)),
    )
    # Пользователь: по согласованию используем блок и подразделение заказчика.
    .withColumn(
        "status_user",
        F.when(
            has_value("model_ver_employer_block_name")
            & has_value("model_ver_employer_dprtmt_name"),
            F.lit(STATUS_OK),
        ).otherwise(F.lit(STATUS_MISSING)),
    )
    .withColumn(
        "status_target_segment",
        status_for_value("model_ver_risk_sgmnt_name", allow_not_applicable=True),
    )
    .withColumn(
        "status_business_process",
        status_for_value("model_ver_busn_proc_code", allow_not_applicable=True),
    )
    .withColumn(
        "status_significance",
        status_for_value("model_ver_signfcnt_ctgry_code", allow_not_applicable=True),
    )
    # Документация: отчет о разработке либо прочие документы версии/модели.
    .withColumn(
        "status_documentation",
        F.when(
            has_value("model_ver_dev_report_sid")
            | has_value("model_ver_other_docs_sid")
            | has_value("model_other_docs_sid"),
            F.lit(STATUS_OK),
        )
        .when(
            is_not_applicable_marker("model_ver_other_docs_sid")
            | is_not_applicable_marker("model_other_docs_sid"),
            F.lit(STATUS_EXCEPTION),
        )
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # Исходный код: ссылка на репозиторий разработки или внедрения.
    .withColumn(
        "status_source_code",
        F.when(
            has_value("model_ver_repstry_link_txt")
            | has_value("model_ver_implm_repstry_link_txt"),
            F.lit(STATUS_OK),
        )
        .when(
            has_value("model_ver_repstry_non_link_reason_name")
            | has_value("model_ver_repstry_non_link_reason_cmnt_txt"),
            F.lit(STATUS_EXCEPTION),
        )
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # Валидация.
    .withColumn(
        "status_validation_result",
        F.when(F.col("validation_result_present") == 1, F.lit(STATUS_OK))
        .when(truthy("model_ver_not_valid_flag"), F.lit(STATUS_EXCEPTION))
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # Ручной мониторинг проверяется отдельно и обязателен по согласованной логике.
    .withColumn(
        "status_manual_monitoring_result",
        F.when(F.col("manual_monitoring_result_present") == 1, F.lit(STATUS_OK))
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # Автомониторинг: отсутствие результата допустимо при явной причине исключения.
    .withColumn(
        "status_auto_monitoring_result",
        F.when(F.col("auto_monitoring_result_present") == 1, F.lit(STATUS_OK))
        .when(
            has_value("model_ver_montrg_auto_exclude_reason_array"),
            F.lit(STATUS_EXCEPTION),
        )
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # Данные валидации.
    .withColumn(
        "status_validation_data",
        F.when(F.col("validation_data_present") == 1, F.lit(STATUS_OK))
        .when(
            truthy("model_ver_not_valid_flag")
            | (F.col("validation_no_sample_exception") == 1),
            F.lit(STATUS_EXCEPTION),
        )
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # Данные ручного мониторинга.
    .withColumn(
        "status_manual_monitoring_data",
        F.when(F.col("manual_monitoring_data_present") == 1, F.lit(STATUS_OK))
        .when(
            F.col("manual_monitoring_no_sample_exception") == 1,
            F.lit(STATUS_EXCEPTION),
        )
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # Данные автоматического мониторинга.
    .withColumn(
        "status_auto_monitoring_data",
        F.when(F.col("auto_monitoring_data_present") == 1, F.lit(STATUS_OK))
        .when(
            (F.col("auto_monitoring_no_sample_exception") == 1)
            | has_value("model_ver_montrg_auto_exclude_reason_array"),
            F.lit(STATUS_EXCEPTION),
        )
        .otherwise(F.lit(STATUS_MISSING)),
    )
    # История изменений модели/версии.
    .withColumn(
        "status_change_history",
        F.when(
            (F.col("version_change_count") > 0)
            | (F.col("model_change_count") > 0),
            F.lit(STATUS_OK),
        ).otherwise(F.lit(STATUS_MISSING)),
    )
    # Нет справочника финансово-значимых параметров.
    .withColumn(
        "status_financially_significant_change_history",
        F.lit(STATUS_NOT_CHECKED),
    )
)

detail = detail.cache()
print("Строк детализации:", detail.count())


## Формирование статистики и Excel

In [ ]:
CHECKS = [
    ("Код и название модели", "status_model_identification"),
    ("Дата разработки / корректировки", "status_model_version_date"),
    ("Дата начала эксплуатации", "status_operation_start"),
    ("Дата окончания эксплуатации", "status_operation_end"),
    ("Владелец модели", "status_owner"),
    ("Разработчик модели", "status_developer"),
    ("Пользователь модели", "status_user"),
    ("Целевой сегмент", "status_target_segment"),
    ("Бизнес-процесс", "status_business_process"),
    ("Степень значимости", "status_significance"),
    ("Документация по модели", "status_documentation"),
    ("Исходный код / скрипт модели", "status_source_code"),
    ("Результаты валидации", "status_validation_result"),
    ("Результаты ручного мониторинга", "status_manual_monitoring_result"),
    ("Результаты автоматического мониторинга", "status_auto_monitoring_result"),
    ("Данные валидации", "status_validation_data"),
    ("Данные ручного мониторинга", "status_manual_monitoring_data"),
    ("Данные автоматического мониторинга", "status_auto_monitoring_data"),
    ("История изменений", "status_change_history"),
    ("История изменений финансово-значимых реквизитов", "status_financially_significant_change_history"),
]

excel_df = detail.limit(MAX_EXCEL_ROWS + 1)

temporal_columns = [
    field.name
    for field in excel_df.schema.fields
    if field.dataType.typeName() in {"date", "timestamp", "timestamp_ntz"}
]
for column_name in temporal_columns:
    excel_df = excel_df.withColumn(column_name, F.col(column_name).cast("string"))

detail_pdf_full = excel_df.toPandas()
if len(detail_pdf_full) > MAX_EXCEL_ROWS:
    raise RuntimeError(
        f"model_required_fields_gaps.xlsx: {len(detail_pdf_full)} строк — превышен лимит Excel"
    )

FLAG_MAP = {
    STATUS_OK: "ОК",
    STATUS_MISSING: "Нарушение",
    STATUS_EXCEPTION: "Не применимо",
    STATUS_NOT_CHECKED: "Не применимо",
}

output_columns = {
    "model_code": "Модель",
    "model_name": "Название",
    "model_ver_num": "Версия",
    "operation_start_dttm": "Дата ввода версии",
    "model_ver_dev_block_name": "Блок разработчика",
    "model_ver_dev_dprtmt_name": "Подразделение разработчика",
    "model_ver_owner_block_name": "Блок владельца",
    "model_ver_owner_dprtmt_name": "Подразделение владельца",
    "model_ver_signfcnt_ctgry_code": "Степень значимости",
}

detail_pdf = detail_pdf_full[list(output_columns.keys())].rename(columns=output_columns)

for label, status_col in CHECKS:
    detail_pdf[label] = detail_pdf_full[status_col].map(FLAG_MAP).fillna("Нарушение")

total_versions = len(detail_pdf_full)
total_models = detail_pdf_full["model_sid"].nunique(dropna=True)

stats_rows = []
for label, status_col in CHECKS:
    mapped = detail_pdf_full[status_col].map(FLAG_MAP)
    violation_mask = mapped.eq("Нарушение")
    ok_mask = mapped.eq("ОК")
    na_mask = mapped.eq("Не применимо")
    applicable_versions = int((~na_mask).sum())
    violations = int(violation_mask.sum())

    stats_rows.append(
        {
            "Проверяемый реквизит": label,
            "Всего версий в эксплуатации": total_versions,
            "Всего моделей в эксплуатации": total_models,
            "Версий ОК": int(ok_mask.sum()),
            "Версий с нарушением": violations,
            "Версий: не применимо": int(na_mask.sum()),
            "% нарушений среди применимых версий": (
                round(violations / applicable_versions * 100, 2)
                if applicable_versions
                else None
            ),
        }
    )

stats_pdf = pd.DataFrame(stats_rows)

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    detail_pdf.to_excel(writer, sheet_name="Детализация", index=False)
    stats_pdf.to_excel(writer, sheet_name="Статистика", index=False)

print("Сохранён отчёт:", OUTPUT_FILE)
print("Строк в детализации:", len(detail_pdf))
print("Столбцов в детализации:", len(detail_pdf.columns))

detail.unpersist()
version_context.unpersist()


## Методологические ограничения

1. `MODEL_VER_EMPLOYER_BLOCK_NAME` и `MODEL_VER_EMPLOYER_DPRTMT_NAME` используются как данные Пользователя модели по согласованной логике, хотя в схеме они обозначены как блок/подразделение заказчика.
2. Для целевого сегмента используется `MODEL_VER_RISK_SGMNT_NAME`.
3. Для даты разработки/корректировки используется `MODEL_VER_CRTN_DTTM`.
4. Для бизнес-процесса проверяется наличие `MODEL_VER_BUSN_PROC_CODE`. В доступной схеме нет однозначного поля, позволяющего напрямую получить название процесса по этому коду.
5. Проверка финансово-значимых реквизитов отдельно невозможна без внешнего перечня соответствующих `ENT_PARAM_CHG_SID`.


In [ ]:
# Закрывайте Spark только если после ноутбука сессия больше не нужна.
# spark.stop()
